# 01 — Prepare Beta-Series Events

**Purpose:** Transform the raw events files into the format needed for beta-series modeling. This is the most important conceptual step that distinguishes beta-series from a basic GLM.

---

## Why this is different from the basic GLM

In the **basic GLM**, all trials of the same condition are pooled together. If there are 8 `positive_valence` feel trials, they all share one regressor. The GLM returns one beta value per condition per voxel.

In **beta-series modeling**, each trial gets its own regressor. The GLM returns one beta map per trial. You end up with a time series of brain maps — one map for each trial — which you can then use to look at how trial-by-trial brain activity correlates across regions (functional connectivity) or with behavior.

---

## What this notebook does

1. Load the raw events TSV files (one per task run)
2. Understand the trial structure: each "feel" trial is preceded by a feedback event that tells you which condition it belongs to
3. Assign each feel trial a **unique label** (e.g., `feel_pos_val_01`, `feel_pos_val_02`, ...) so the GLM treats it as its own regressor
4. Save the labeled events TSVs to the beta-series output folder

**Output:** Two labeled events TSV files (one per run), saved under `Analysis/outputs/{subject}/beta_series/`

---
## Section 1 — Configuration and Paths

Set the subject ID here and derive all paths from it. The two directories you need to define are:

**Input — raw events files:**
```
C:\ManzaRotation\Raw\sub-097\func\
```

**Output — where labeled events will be saved:**
```
C:\ManzaRotation\Analysis\outputs\sub-097\beta_series\
```

Setting up the paths and creating the output directory looks like this:

```python
project_dir  = Path(r"C:\ManzaRotation")
raw_func_dir = project_dir / "Raw" / subject / "func"
out_dir      = project_dir / "Analysis" / "outputs" / subject / "beta_series"

out_dir.mkdir(parents=True, exist_ok=True)  # creates the folder if it doesn't exist yet
```

**TODO:**
- [ ] Set `subject` and `tasks`
- [ ] Define `raw_func_dir` and `out_dir` using the pattern above
- [ ] Call `.mkdir(parents=True, exist_ok=True)` on `out_dir`

In [ ]:
from pathlib import Path
import pandas as pd

subject = "sub-097"
tasks = ["modulate1", "modulate2"]

# TODO: define raw_func_dir and out_dir, then create out_dir


---
## Section 2 — Load and Inspect the Raw Events Files

Load both events TSVs into pandas DataFrames and look at their structure. The two files you are loading are:

```
C:\ManzaRotation\Raw\sub-097\func\sub-097_task-modulate1_events.tsv
C:\ManzaRotation\Raw\sub-097\func\sub-097_task-modulate2_events.tsv
```

Loading a TSV with pandas:

```python
events1 = pd.read_csv(raw_func_dir / f"{subject}_task-modulate1_events.tsv", sep="\t")
events2 = pd.read_csv(raw_func_dir / f"{subject}_task-modulate2_events.tsv", sep="\t")
```

Then inspect the structure — these two calls will tell you most of what you need to know:

```python
print(events1.head(20))                        # see the feedback→feel row pattern
print(events1["trial_type"].value_counts())    # how many of each event type
```

**TODO:**
- [ ] Load `events1` and `events2`
- [ ] Print the first 20 rows and value counts for `events1`
- [ ] Confirm the feedback→feel alternating pattern is visible

In [ ]:
# TODO: load events1 and events2


In [ ]:
# TODO: inspect events1 — show first 20 rows and value_counts


---
## Section 3 — Understand the Trial Structure

The task alternates between feedback events and feel events. The feedback event tells you what kind of feedback the participant just received. The feel event that immediately follows is the period you want to model.

**The four feedback types and what they mean:**

| `trial_type` in raw events | Meaning |
|---|---|
| `fb_v_pos` | Feedback: positive valence |
| `fb_v_neg` | Feedback: negative valence |
| `fb_a_pos` | Feedback: positive arousal |
| `fb_a_neg` | Feedback: negative arousal |

To trace the feedback→feel pairs, look at adjacent rows using `.iloc`:

```python
# Look at two adjacent rows by position:
print(events1.iloc[0])   # row 0 — should be a feedback event (fb_v_pos, etc.)
print(events1.iloc[1])   # row 1 — should be the feel event that follows it

# Or look at several pairs at once:
print(events1[["trial_type"]].head(10))
```

Your goal: for each `feel` row, look at the row immediately before it (at `idx - 1`) to determine which condition it belongs to.

**TODO:**
- [ ] Look at the printed rows from Section 2 and trace the feedback→feel pairs
- [ ] Confirm every `feel` row is preceded by one of the four `fb_*` rows
- [ ] Decide on your naming convention for unique labels and write it in the notes cell below

*Your notes on naming convention:*

> (write here before coding)

---
## Section 4 — Label Each Feel Trial with a Unique Identifier

Write a function that takes a raw events DataFrame and returns a new one where each `feel` row has a unique `trial_type`. The key challenge is keeping a per-condition counter as you loop through the rows.

The counter pattern looks like this:

```python
counters = {"pos_val": 0, "neg_val": 0, "pos_aro": 0, "neg_aro": 0}

for idx in events.index:
    if events.loc[idx, "trial_type"] == "feel":
        prev_type = events.loc[idx - 1, "trial_type"]   # look at the row before
        condition = fb_to_condition[prev_type]            # e.g. "pos_val"
        counters[condition] += 1
        label = f"{condition}_{counters[condition]:02d}"  # e.g. "pos_val_01"
        # assign label to the row in your working copy
```

Your function `make_beta_events(events)` should wrap this logic and return a clean DataFrame with only `onset`, `duration`, and `trial_type` columns, keeping only the `feel` rows.

**TODO:**
- [ ] Define `fb_to_condition` dict mapping `fb_v_pos` → `pos_val`, etc.
- [ ] Write `make_beta_events(events)` using the counter pattern above
- [ ] Apply it to `events1` and `events2` and print the results

In [ ]:
# Map from raw feedback trial_type to a short condition name
fb_to_condition = {
    "fb_v_pos": "pos_val",
    "fb_v_neg": "neg_val",
    "fb_a_pos": "pos_aro",
    "fb_a_neg": "neg_aro",
}

# TODO: write make_beta_events(events) -> labeled feel-only DataFrame
def make_beta_events(events):
    """
    Takes a raw events DataFrame and returns a new DataFrame
    containing only feel trials, each with a unique trial_type label.
    """
    # TODO: implement
    pass


In [ ]:
# TODO: apply make_beta_events to events1 and events2, inspect the results
# beta_events1 = make_beta_events(events1)
# beta_events2 = make_beta_events(events2)
# print(beta_events1)


---
## Section 5 — Verify the Labeled Events

Before saving, double-check that the labeled events are structured correctly. These checks should all pass:

```python
# Shape: should be (32, 3) — 32 feel trials, 3 columns
print(beta_events1.shape)

# Uniqueness: all 32 labels should be different
print(beta_events1["trial_type"].nunique())        # should print 32

# Each label appears exactly once (count of 1 for every row)
print(beta_events1["trial_type"].value_counts())

# Spot-check: print the full DataFrame to eyeball onsets/durations
print(beta_events1)
```

**TODO:**
- [ ] Run all four checks above for `beta_events1`
- [ ] Repeat for `beta_events2`
- [ ] If `nunique()` is less than 32, there's a bug in your counter logic — go back to Section 4

In [ ]:
# TODO: verification checks on beta_events1 and beta_events2


---
## Section 6 — Save the Labeled Events

Save the labeled events DataFrames to the beta-series output directory. The two output files should end up at:

```
C:\ManzaRotation\Analysis\outputs\sub-097\beta_series\sub-097_task-modulate1_events-betaSeries.tsv
C:\ManzaRotation\Analysis\outputs\sub-097\beta_series\sub-097_task-modulate2_events-betaSeries.tsv
```

Saving a DataFrame as a TSV:

```python
out_path1 = out_dir / f"{subject}_task-modulate1_events-betaSeries.tsv"
beta_events1.to_csv(out_path1, sep="\t", index=False)
print(out_path1.exists())  # should print True
```

**TODO:**
- [ ] Save `beta_events1` and `beta_events2` using this pattern
- [ ] Confirm both output files exist

In [ ]:
# TODO: save beta_events1 and beta_events2, confirm files exist
